# Modul B · Kapitel 2.1 — Knowledge Base und Chunking

## Challenge: Aus Dokumenten werden durchsuchbare Chunks


**Lernziel:** Du kannst eine Knowledge Base aufbauen, Chunking-Verfahren implementieren und
vergleichen, und du legst die Chunks mit ihren Metadaten in einer Vector Database ab.

Dieses Notebook baut den linken Teil der RAG-Kette:

```
Dokumente ──► Chunks ──► Embeddings ──► Vector Database ──► Retrieval ──► Prompt ──► Antwort
    └────────── hier ──────────────────────────┘
```

Die Knowledge Base ist die Wissensbasis eines Security Operations Center: CVE-Advisories,
Runbooks, Richtlinien, ein Post-Mortem und eine Systemdokumentation.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind insgesamt **5 Challenges**.

---
## 0 · Setup

▶️ Führe die beiden nächsten Zellen aus.

Gerechnet wird lokal mit **Ollama**: `qwen3.5:0.8b` beantwortet Fragen, `nomic-embed-text`
erzeugt Embeddings. Beides läuft über denselben OpenAI-Client.

Die Naht zum Modell sitzt an genau einer Stelle, in `helfer.py`:

```python
# Ollama, lokal. Für einen Provider: base_url und api_key tauschen, Modellname anpassen.
BASIS_URL = "http://localhost:11434/v1"
API_KEY = "ollama"                # Ollama prüft den Key nicht
MODELL = "qwen3.5:0.8b"           # 1,0 GB, läuft auf jedem Laptop
EMBEDDING_MODELL = "nomic-embed-text"
REASONING = "none"                # qwen3.5 denkt sonst und liefert leeren content

client = OpenAI(base_url=BASIS_URL, api_key=API_KEY)
```

Das Notebook **importiert** diese Namen, statt sie noch einmal zu setzen. Sonst stünde das
Modell an zwei Stellen, und ein Wechsel würde an einer davon nicht greifen.

In Google Colab gibt es kein lokales Ollama. Dort trägst du in `helfer.py` Adresse und Key
deines Providers ein und passt die beiden Modellnamen an.

In [ ]:
# ▶️ Pakete, helfer.py und die Verbindung zum Modell
import json
import re
import statistics
import sys
from pathlib import Path

try:
    import chromadb
    import tiktoken
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    %pip install -q chromadb tiktoken langchain-text-splitters
    import chromadb
    import tiktoken
    from langchain_text_splitters import RecursiveCharacterTextSplitter

import matplotlib.pyplot as plt

# helfer.py liegt im Ordner dieses Notebooks
for kandidat in [Path.cwd(), *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer
from helfer import client, frage_llm, MODELL, EMBEDDING_MODELL

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

print(f"chromadb {chromadb.__version__}")
print(f"Modell: {MODELL} · Embeddings: {EMBEDDING_MODELL} · Server: {client.base_url}")
print("Setup fertig ✔")

▶️ Die zweite Zelle lädt die Wissensbasis. Die Funktionen, die alle Notebooks dieses Kapitels
teilen, stehen in `helfer.py` — darunter `frage_llm()`, `embed()` und `zaehle_tokens()`.

In [ ]:
# ▶️ Wissensbasis und Evaluationsfragen laden
dokumente = helfer.lade_dokumente()
fragen = helfer.lade_fragen()

print(f"{len(dokumente)} Dokumente, {len(fragen)} Evaluationsfragen")
print()
for d in dokumente:
    print(f"  {d['id']:<34} {d['titel'][:60]}")

---
## 1 · Die Knowledge Base

📖 Eine Knowledge Base entsteht nicht auf einmal. Sie wird aus den Quellen zusammengetragen, die
im Unternehmen ohnehin existieren:

| Quelle | Beispiel | Besonderheit |
|---|---|---|
| Dokumentation | Runbooks, Systemdoku, Richtlinien | gepflegt, aber unterschiedlich aktuell |
| Ticketsystem | gelöste Vorfälle, Changes | viel Text, wenig Struktur |
| Herstellerangaben | CVE-Advisories, Release Notes | strukturiert, ändern sich nachträglich |
| Post-Mortems | Auswertungen von Vorfällen | dicht, hoher Wert je Satz |
| Logs und Inventare | Asset-Inventar, SIEM-Auszüge | maschinell, sehr groß |

Vier Schritte führen von dort zur Knowledge Base: **einsammeln**, in Text **umwandeln**
(PDF, HTML, Confluence, Tickets), **bereinigen** (Navigationsleisten, Seitenzahlen, doppelte
Dokumente) und **zerlegen** — das Chunking.

Unsere Wissensbasis liegt schon als Markdown in `daten/wissensbasis/`. Schauen wir sie uns an.

In [ ]:
# ▶️ Länge und Struktur je Dokument
print(f"{'Dokument':<36}{'Zeichen':>9}{'Wörter':>8}{'Tokens':>8}{'Überschriften':>14}{'Tabellen':>10}")
print("-" * 85)

gesamt_tokens = 0
for d in dokumente:
    text = d["text"]
    tokens = helfer.zaehle_tokens(text)
    gesamt_tokens += tokens
    ueberschriften = len(re.findall(r"(?m)^#{1,3} ", text))
    tabellen = len(re.findall(r"(?m)^\|---", text))
    print(f"{d['id']:<36}{len(text):>9,}{len(text.split()):>8,}{tokens:>8,}"
          f"{ueberschriften:>14}{tabellen:>10}".replace(",", "."))

print("-" * 85)
print(f"{'gesamt':<36}{'':>9}{'':>8}{gesamt_tokens:>8,}".replace(",", "."))

📖 Zwei Dinge fallen auf.

**Die Längen gehen weit auseinander.** Das kürzeste Dokument hat rund 2.400 Zeichen, das längste
über 17.000 — Faktor sieben. Ein Verfahren, das alle Dokumente gleich behandelt, muss mit beidem
zurechtkommen.

**Jedes Dokument hat Struktur.** Überschriften, Tabellen, Listen. Diese Struktur ist eine
Vorlage für sinnvolle Schnitte, und sie geht verloren, wenn man stur nach Zeichen zählt.

In [ ]:
# ▶️ Die Gliederung des längsten Dokuments
lang = next(d for d in dokumente if d["id"] == "systemdoku-sentinelgrid")

for zeile in lang["text"].splitlines():
    if re.match(r"^#{1,3} ", zeile):
        tiefe = len(zeile) - len(zeile.lstrip("#"))
        print("   " * (tiefe - 1) + zeile.lstrip("# "))

---
## 2 · Warum Chunks

📖 Der naheliegende Weg wäre: die ganze Wissensbasis in den Prompt schreiben und fragen. Drei
Gründe sprechen dagegen.

1. **Das Context Window ist begrenzt.** Was nicht hineinpasst, wird abgeschnitten — ohne
   Fehlermeldung. Das Modell antwortet trotzdem, nur eben ohne die abgeschnittene Stelle.
2. **Tokens kosten.** Abgerechnet wird je Token. Wer bei jeder Frage die gesamte Wissensbasis
   mitschickt, bezahlt sie bei jeder Frage.
3. **Viel Kontext verwässert.** Je mehr Text im Prompt steht, desto mehr Stellen sehen nach
   einer Antwort aus. Genau das führt zu falschen Antworten mit hohem Selbstbewusstsein.

▶️ Wir stellen zwei Fragen, deren Antwort in der Wissensbasis steht — einmal mit allem, einmal
mit dem passenden Ausschnitt.

In [ ]:
# ▶️ Versuch A: die gesamte Wissensbasis als Kontext
SYSTEM = ("Du bist ein Assistent im Security Operations Center. "
          "Antworte kurz und nur mit dem, was im Kontext steht.")

alles = "\n\n---\n\n".join(d["text"] for d in dokumente)

TESTFRAGEN = [
    "Innerhalb welcher Frist muss ein Notfall-Patch auf Servern eingespielt werden?",
    "Wie lange werden Proxy- und DNS-Logs aufbewahrt?",
]

print(f"Kontext: {helfer.zaehle_tokens(alles):,} Tokens".replace(",", "."))
print()
for frage in TESTFRAGEN:
    antwort = frage_llm(f"Kontext:\n{alles}\n\nFrage: {frage}\nAntwort:",
                               system=SYSTEM, max_tokens=120)
    print(f"F: {frage}")
    print(f"A: {' '.join(antwort.split())[:300]}")
    print()

In [ ]:
# ▶️ Versuch B: nur der passende Ausschnitt
def ausschnitt(dok_id, ab, laenge=800):
    """Schneidet ab der Überschrift `ab` ein Stück aus einem Dokument."""
    text = next(d["text"] for d in dokumente if d["id"] == dok_id)
    start = text.index(ab)
    return text[start:start + laenge]


AUSSCHNITTE = [
    ausschnitt("runbook-patch-management", "## Fristen"),
    ausschnitt("policy-logging-und-aufbewahrung", "## Aufbewahrungsfristen"),
]

for frage, kontext in zip(TESTFRAGEN, AUSSCHNITTE):
    antwort = frage_llm(f"Kontext:\n{kontext}\n\nFrage: {frage}\nAntwort:",
                               system=SYSTEM, max_tokens=120)
    print(f"Kontext: {helfer.zaehle_tokens(kontext)} Tokens")
    print(f"F: {frage}")
    print(f"A: {' '.join(antwort.split())[:300]}")
    print()

📖 Die richtigen Antworten lauten **72 Stunden** (Runbook Patch-Management, Abschnitt Fristen)
und **90 Tage** (Richtlinie Logging, Zeile Proxy- und DNS-Logs).

Mit dem Ausschnitt stehen genau diese beiden Werte da. Mit der gesamten Wissensbasis steht
keiner von beiden: Im Durchlauf oben erklärt das Modell einmal, es gebe keine Frist, und
zitiert einmal einen Abschnitt der Systemdokumentation, in dem es um etwas anderes geht.

Die Ursache ist dieselbe. Der Prompt ist größer als das Context Window des Servers, der Rest
fällt ohne Fehlermeldung weg, und was übrig bleibt, enthält Dutzende ähnlich aussehender
Stellen — 72 Stunden Meldefrist, vier Stunden Wiederanlauf, 90 Tage Aufbewahrung. Beachte die
Form der Fehler: keine erfundene Zahl, sondern eine Antwort mit Abschnittsangabe. Falsche
Antworten aus zu viel Kontext sehen aus wie belegte Antworten.

Die Rechnung dazu:

| | ganze Wissensbasis | ein Ausschnitt |
|---|---:|---:|
| Tokens je Frage | rund 15.000 | rund 250 |
| bei 200 Fragen am Tag | 3.000.000 | 50.000 |
| Faktor | 60 | 1 |

Und diese Wissensbasis hat elf Dokumente. Eine echte hat zehntausend.

Damit ist die Aufgabe gestellt: Wir brauchen **Ausschnitte, die für sich stehen** — Chunks. Ein
Chunk ist ein Stück Text, das klein genug für den Prompt ist und groß genug, um eine Frage
allein zu beantworten.

---
## 3 · Verfahren 1: feste Länge

📖 Das einfachste Chunking schneidet den Text alle *n* Zeichen durch. Damit ein Satz an der
Schnittstelle nicht verloren geht, überlappen benachbarte Chunks um ein Stück:

```
Text:    ────────────────────────────────────────────────
Chunk 0: ──────────────
Chunk 1:          ──────────────
Chunk 2:                   ──────────────
                  └──┘ Überlappung
```

Zwei Einstellungen: `groesse` ist die Länge eines Chunks, `ueberlappung` der gemeinsame Teil
mit dem nächsten. Das Fenster rückt also um `groesse - ueberlappung` Zeichen weiter.

Das Verfahren kennt keine Sätze, keine Absätze und keine Tabellen. Dafür ist es in zehn Zeilen
geschrieben und funktioniert auf jedem Text.

### 🛠️ Challenge 1: Chunking mit fester Länge

Schreibe `chunke_fest(text, groesse, ueberlappung)`. Die Funktion gibt eine **Liste von
Textstücken** zurück:

* jedes Stück ist höchstens `groesse` Zeichen lang, nur das letzte darf kürzer sein,
* zwei benachbarte Stücke teilen sich die letzten beziehungsweise ersten `ueberlappung` Zeichen,
* kein Zeichen des Textes geht verloren.

*Tipp: Die Schrittweite ist `groesse - ueberlappung`. `text[start:start + groesse]` schneidet
ein Stück heraus; über das Ende hinaus zu schneiden ist in Python erlaubt und liefert einfach
weniger.*

In [ ]:
def chunke_fest(text, groesse=800, ueberlappung=100):
    """Zerlegt einen Text in Stücke fester Zeichenlänge mit Überlappung."""
    if groesse <= 0 or not 0 <= ueberlappung < groesse:
        raise ValueError("Es muss 0 <= ueberlappung < groesse gelten")

    # TODO 1: Um wie viele Zeichen rückt das Fenster weiter?
    schritt = ...

    stuecke = []
    start = 0
    while start < len(text):
        # TODO 2: das Stück ab `start` mit der Länge `groesse` anhängen
        ...
        if start + groesse >= len(text):
            break
        # TODO 3: das Fenster weiterschieben
        ...

    return stuecke


In [ ]:
# ✅ Selbsttest
probe = "".join(str(i % 10) for i in range(1000))
stuecke = chunke_fest(probe, 100, 20)

assert all(len(s) <= 100 for s in stuecke), "Kein Chunk darf länger als groesse sein"
assert all(len(s) == 100 for s in stuecke[:-1]), "Nur der letzte Chunk darf kürzer sein"
assert stuecke[0][-20:] == stuecke[1][:20], "Die Überlappung muss am Anfang des nächsten Chunks stehen"
assert stuecke[0] + "".join(s[20:] for s in stuecke[1:]) == probe, "Kein Zeichen darf verloren gehen"
assert len(stuecke) == 13, f"13 Chunks erwartet, {len(stuecke)} bekommen"
assert chunke_fest("kurzer Text", 100, 20) == ["kurzer Text"], "Kurzer Text ergibt genau einen Chunk"

print("✅ Challenge 1 gelöst")
print(f"1.000 Zeichen, groesse=100, ueberlappung=20  →  {len(stuecke)} Chunks")
print(f"  Chunk 0 endet auf: {stuecke[0][-20:]!r}")
print(f"  Chunk 1 beginnt mit: {stuecke[1][:20]!r}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def chunke_fest(text, groesse=800, ueberlappung=100):
    """Zerlegt einen Text in Stücke fester Zeichenlänge mit Überlappung."""
    if groesse <= 0 or not 0 <= ueberlappung < groesse:
        raise ValueError("Es muss 0 <= ueberlappung < groesse gelten")

    schritt = groesse - ueberlappung

    stuecke = []
    start = 0
    while start < len(text):
        stuecke.append(text[start:start + groesse])
        if start + groesse >= len(text):
            break
        start += schritt

    return stuecke
```

Ohne das `break` würde bei einem Text, dessen Länge genau aufgeht, ein leerer Chunk am Ende
entstehen.

</details>

In [ ]:
# ▶️ Das Verfahren auf einem echten Dokument
patch = next(d for d in dokumente if d["id"] == "runbook-patch-management")
feste_chunks = chunke_fest(patch["text"], 800, 100)

helfer.zeige_chunks(feste_chunks, n=2)

📖 Der erste Chunk endet mitten in der Tabelle der Fristen, der zweite beginnt mitten darin. Wer
später nach der Frist für Notfall-Patches sucht, findet einen Chunk, in dem die Kopfzeile der
Tabelle fehlt — und damit die Bedeutung der Spalten.

---
## 4 · Verfahren 2: an der Struktur entlang

📖 Markdown-Dokumente sagen selbst, wo sie zusammengehören: Überschriften beginnen einen
Abschnitt, Leerzeilen trennen Absätze. Ein Chunking, das diese Grenzen benutzt, schneidet keine
Sätze und keine Tabellen durch.

Zwei Grenzen braucht es trotzdem:

* **Maximalgröße.** Ein Abschnitt kann sehr lang sein. Wird er zu lang, muss auch mitten im
  Abschnitt getrennt werden.
* **Mindestgröße.** Eine Überschrift mit einem Satz darunter ergibt einen Chunk aus zwölf
  Wörtern. Solche Schnipsel liefern später schlechte Embeddings, weil zu wenig Inhalt drinsteht.
  Sie werden mit dem nächsten Stück zusammengelegt.

### 🛠️ Challenge 2: Chunking an der Dokumentstruktur

Jetzt kommt nur eine neue Entscheidung hinzu: Ein Chunk soll möglichst an einer Überschrift
oder Absatzgrenze enden. Die Größenregeln verhindern sehr kleine oder zu große Chunks.

Vervollständige chunke_nach_struktur(). Das Codegerüst führt durch den Ablauf:

1. Text an Leerzeilen in Absätze zerlegen.
2. Vor jedem Absatz prüfen, ob der aktuelle Chunk dadurch zu lang würde.
3. Auch an einer Überschrift neu beginnen, wenn der aktuelle Chunk schon groß genug ist.
4. Einzelne überlange Absätze mit chunke_fest() zerlegen.
5. Kleine Nachbar-Chunks zusammenführen, sofern sie gemeinsam ins Maximum passen.

Bearbeite zuerst die drei TODO-Stellen im Hauptdurchlauf. Die letzte Schleife wendet dieselbe
Größenprüfung noch einmal auf kleine Chunks an.


In [ ]:
def chunke_nach_struktur(text, max_zeichen=1200, min_zeichen=300):
    """Trennt an Überschriften und Absätzen, hält min_zeichen und max_zeichen ein."""
    # TODO 1: Text an Leerzeilen zerlegen, leere Absätze wegwerfen
    abschnitte = ...

    roh = []
    aktuell = ""
    for absatz in abschnitte:
        ist_ueberschrift = absatz.startswith("#")
        # TODO 2: Käme der aktuelle Chunk mit diesem Absatz über max_zeichen?
        zu_lang = ...

        if aktuell and (zu_lang or (ist_ueberschrift and len(aktuell) >= min_zeichen)):
            roh.append(aktuell)
            aktuell = ""

        if len(absatz) > max_zeichen:
            if aktuell:
                roh.append(aktuell)
                aktuell = ""
            teile = chunke_fest(absatz, max_zeichen, 0)
            roh.extend(teile[:-1])
            aktuell = teile[-1]
        else:
            # TODO 3: Absatz an den aktuellen Chunk hängen, getrennt durch eine Leerzeile
            aktuell = ...

    if aktuell:
        roh.append(aktuell)

    chunks = []
    for stueck in roh:
        # TODO 4: zu kurzen Chunk mit dem folgenden zusammenlegen, wenn max_zeichen es zulässt
        if chunks and ...:
            chunks[-1] = f"{chunks[-1]}\n\n{stueck}"
        else:
            chunks.append(stueck)

    return chunks


In [ ]:
# ✅ Selbsttest
BEISPIEL = (
    "# Titel des Dokuments\n\n"
    + "Ein einleitender Absatz, der lang genug ist, um die Mindestgröße zu erreichen. " * 3
    + "\n\n## Zweiter Abschnitt\n\n"
    + "Der Text des zweiten Abschnitts, ebenfalls ausreichend lang für einen eigenen Chunk. " * 3
)

teile = chunke_nach_struktur(BEISPIEL, max_zeichen=400, min_zeichen=100)
assert len(teile) == 2, f"Zwei Chunks erwartet, {len(teile)} bekommen"
assert teile[0].startswith("# Titel des Dokuments"), "Der erste Chunk beginnt mit der Überschrift"
assert teile[1].startswith("## Zweiter Abschnitt"), "Die zweite Überschrift beginnt einen neuen Chunk"

echte = chunke_nach_struktur(patch["text"])
assert max(len(c) for c in echte) <= 1200, "max_zeichen wird verletzt"
assert all(c.strip() for c in echte), "Leere Chunks darf es nicht geben"
assert all(c.startswith("#") for c in echte), "Jeder Chunk dieses Dokuments beginnt mit einer Überschrift"
for absatz in re.split(r"\n\s*\n", patch["text"]):
    if absatz.strip():
        assert any(absatz.strip() in c for c in echte), f"Absatz verloren: {absatz[:40]!r}"

print("✅ Challenge 2 gelöst")
print(f"{patch['id']}:  {len(feste_chunks)} feste Chunks  →  {len(echte)} Struktur-Chunks")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def chunke_nach_struktur(text, max_zeichen=1200, min_zeichen=300):
    """Trennt an Überschriften und Absätzen, hält min_zeichen und max_zeichen ein."""
    abschnitte = [a.strip() for a in re.split(r"\n\s*\n", text) if a.strip()]

    roh = []
    aktuell = ""
    for absatz in abschnitte:
        ist_ueberschrift = absatz.startswith("#")
        zu_lang = len(aktuell) + len(absatz) + 2 > max_zeichen

        if aktuell and (zu_lang or (ist_ueberschrift and len(aktuell) >= min_zeichen)):
            roh.append(aktuell)
            aktuell = ""

        if len(absatz) > max_zeichen:
            if aktuell:
                roh.append(aktuell)
                aktuell = ""
            teile = chunke_fest(absatz, max_zeichen, 0)
            roh.extend(teile[:-1])
            aktuell = teile[-1]
        else:
            aktuell = f"{aktuell}\n\n{absatz}" if aktuell else absatz

    if aktuell:
        roh.append(aktuell)

    chunks = []
    for stueck in roh:
        if chunks and len(chunks[-1]) < min_zeichen \
                and len(chunks[-1]) + len(stueck) + 2 <= max_zeichen:
            chunks[-1] = f"{chunks[-1]}\n\n{stueck}"
        else:
            chunks.append(stueck)

    return chunks
```

Der Nachlauf am Ende ist wichtig: Ohne ihn entstehen aus kurzen Abschnitten wie
`## Referenzen` Chunks aus zwei Zeilen.

</details>

In [ ]:
# ▶️ Dasselbe Dokument, das andere Verfahren
struktur_chunks = chunke_nach_struktur(patch["text"])

helfer.zeige_chunks(struktur_chunks, n=2)

---
## 5 · Die Verfahren vergleichen

📖 „Sieht besser aus" ist keine Bewertung. Vier Kennzahlen machen den Unterschied messbar:

| Kennzahl | Was sie zeigt |
|---|---|
| **Anzahl** | wie viele Einträge die Vector Database bekommt |
| **Längenverteilung** | kürzester, mittlerer, längster Chunk |
| **zerschnitten** | wie oft ein Chunk mitten im Satz endet |
| **Belege** | wie oft eine bekannte Antwort vollständig in einem Chunk liegt |

Die letzte Kennzahl ist die wichtigste. Eine Antwort, die auf zwei Chunks verteilt ist, kommt im
Retrieval nur halb an — und eine halbe Frist ist schlimmer als keine.

▶️ Die nächste Zelle schneidet fünf Belegstellen aus der Wissensbasis. Jede beantwortet eine der
Fragen aus `daten/fragen.json` vollständig.

In [ ]:
# ▶️ Fünf Belegstellen: die Textstücke, die eine Frage vollständig beantworten
def belegstelle(dok_id, ab, bis):
    """Schneidet den Text zwischen zwei Marken aus einem Dokument."""
    text = next(d["text"] for d in dokumente if d["id"] == dok_id)
    start = text.index(ab)
    return text[start:text.index(bis, start)].strip()


BELEGE = [
    belegstelle("cve-2026-3224", "## Bewertung", "## Betroffene Versionen"),
    belegstelle("runbook-patch-management", "## Fristen", "## Regelweg"),
    belegstelle("policy-logging-und-aufbewahrung", "## Aufbewahrungsfristen", "## Zeit"),
    belegstelle("systemdoku-sentinelgrid", "## 3 Kapazität", "Die Antwortzeit"),
    belegstelle("runbook-zugriffskontrolle", "## Austritt", "Bei einer fristlosen"),
]

SATZENDE = (".", "!", "?", ":", "|", "-", ")")

for b in BELEGE:
    print(f"{len(b):>4} Zeichen  {b.splitlines()[0]}")

### 🛠️ Challenge 3: Die Bewertungsfunktion

Schreibe `bewerte(chunks)`. Eingabe ist eine Liste von Textstücken, Ausgabe ein Dict mit genau
diesen Schlüsseln:

| Schlüssel | Wert |
|---|---|
| `anzahl` | Zahl der Chunks |
| `kuerzester`, `median`, `laengster` | Chunklänge in Zeichen, der Median als `int` |
| `zerschnitten` | Zahl der Chunks, die **nicht** auf ein Zeichen aus `SATZENDE` enden — der letzte Chunk zählt nicht mit |
| `belege` | wie viele der Einträge aus `BELEGE` vollständig in **einem** Chunk stehen |

*Tipp: `statistics.median(...)`, `str.rstrip()` und `str.endswith(SATZENDE)` — `endswith`
nimmt ein Tupel. Für `belege`: `any(b in c for c in chunks)`.*

In [ ]:
def bewerte(chunks):
    """Kennzahlen einer Chunk-Liste: Anzahl, Längen, Schnitte, gefundene Belegstellen."""
    laengen = [len(c) for c in chunks]

    return {
        "anzahl": ...,
        "kuerzester": ...,
        "median": ...,
        "laengster": ...,
        # TODO: Chunks, die nicht auf ein Satzende enden — ohne den letzten
        "zerschnitten": ...,
        # TODO: wie viele Belegstellen vollständig in einem Chunk stehen
        "belege": ...,
    }


In [ ]:
# ✅ Selbsttest
w = bewerte(["Ein vollständiger Satz.", "Ein Chunk der mitten im Satz", "Und Schluss."])
assert set(w) == {"anzahl", "kuerzester", "median", "laengster", "zerschnitten", "belege"}, \
    "Die Schlüssel stimmen nicht"
assert w["anzahl"] == 3
assert (w["kuerzester"], w["laengster"]) == (12, 28), "Längen in Zeichen"
assert w["median"] == 23 and isinstance(w["median"], int), "Median als ganze Zahl"
assert w["zerschnitten"] == 1, "Nur der mittlere Chunk endet mitten im Satz"
assert w["belege"] == 0, "In diesen drei Sätzen steht keine Belegstelle"

echte_bewertung = bewerte(chunke_nach_struktur(patch["text"]))
assert echte_bewertung["laengster"] <= 1200

print("✅ Challenge 3 gelöst")
print(w)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def bewerte(chunks):
    """Kennzahlen einer Chunk-Liste: Anzahl, Längen, Schnitte, gefundene Belegstellen."""
    laengen = [len(c) for c in chunks]

    return {
        "anzahl": len(chunks),
        "kuerzester": min(laengen),
        "median": int(statistics.median(laengen)),
        "laengster": max(laengen),
        "zerschnitten": sum(1 for c in chunks[:-1] if not c.rstrip().endswith(SATZENDE)),
        "belege": sum(1 for b in BELEGE if any(b in c for c in chunks)),
    }
```

</details>

In [ ]:
# ▶️ Beide Verfahren über die gesamte Wissensbasis
def ueber_alle(verfahren):
    """Wendet ein Chunking-Verfahren auf jedes Dokument an und hängt die Ergebnisse aneinander."""
    stuecke = []
    for d in dokumente:
        stuecke += verfahren(d["text"])
    return stuecke


VERFAHREN = {
    "fest 800/100": lambda t: chunke_fest(t, 800, 100),
    "fest 1200/150": lambda t: chunke_fest(t, 1200, 150),
    "Struktur": chunke_nach_struktur,
}

ergebnisse = {name: ueber_alle(f) for name, f in VERFAHREN.items()}

kopf = f"{'Verfahren':<16}{'Anzahl':>8}{'kürzest':>9}{'Median':>8}{'längst':>8}{'zerschnitten':>14}{'Belege':>9}"
print(kopf)
print("-" * len(kopf))
for name, stuecke in ergebnisse.items():
    w = bewerte(stuecke)
    print(f"{name:<16}{w['anzahl']:>8}{w['kuerzester']:>9}{w['median']:>8}"
          f"{w['laengster']:>8}{w['zerschnitten']:>14}{str(w['belege']) + '/5':>9}")

In [ ]:
# ▶️ Die Längenverteilung im Diagramm
plt.figure(figsize=(8, 4.5))
plt.hist([[len(c) for c in ergebnisse["fest 800/100"]],
          [len(c) for c in ergebnisse["Struktur"]]],
         bins=range(0, 1400, 100), color=[ORANGE, BLAU],
         label=["fest 800/100", "Struktur"])
plt.xlabel("Chunklänge in Zeichen")
plt.ylabel("Zahl der Chunks")
plt.title("Feste Länge erzeugt eine Säule, Struktur eine Verteilung")
plt.legend()
plt.show()

📖 Das feste Verfahren erzeugt fast nur Chunks von genau `groesse` Zeichen — die eine Säule im
Diagramm. Die Reste am Dokumentende sind der kleine Rest links.

Beim Struktur-Verfahren streuen die Längen zwischen der Mindest- und der Maximalgröße. Das ist
kein Mangel, sondern das Ziel: Die Länge richtet sich nach dem Abschnitt, nicht nach einer
Zahl.

Entscheidend sind die beiden rechten Spalten der Tabelle. Das feste Verfahren zerschneidet die
große Mehrheit seiner Chunks mitten im Satz und liefert zwei von fünf Belegstellen vollständig.
Das Struktur-Verfahren schneidet fast nie mitten im Satz und liefert alle fünf.

Ein größeres `groesse` hilft nicht. Es verschiebt die Schnitte nur an andere Stellen.

---
## 6 · Dasselbe mit dem Framework

📖 Genau dieses Verfahren gibt es fertig. Der `RecursiveCharacterTextSplitter` aus
`langchain_text_splitters` arbeitet mit einer **Liste von Trennzeichen** und geht sie der Reihe
nach durch:

1. Er versucht, den Text am ersten Trennzeichen zu teilen (`"\n## "`, also an Überschriften).
2. Ist ein Stück danach immer noch länger als `chunk_size`, nimmt er das nächste Trennzeichen
   (`"\n\n"`, Absätze), dann `"\n"`, dann `". "`, dann `" "`.
3. Das letzte Trennzeichen ist der leere String — dort wird hart geschnitten. Das ist genau
   `chunke_fest`.

Die vier Parameter:

| Parameter | Bedeutung |
|---|---|
| `chunk_size` | Maximalgröße, gemessen mit `length_function` |
| `chunk_overlap` | Überlappung, wenn hart geschnitten werden muss |
| `separators` | die Trennzeichen, vom besten zum schlechtesten |
| `length_function` | `len` misst Zeichen; mit `helfer.zaehle_tokens` misst man Tokens |

▶️ Wir stellen ihn auf dieselben Grenzen ein wie unser eigenes Verfahren.

In [ ]:
# ▶️ Das Framework, mit denselben Grenzen
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

ergebnisse["LangChain"] = ueber_alle(splitter.split_text)

kopf = f"{'Verfahren':<16}{'Anzahl':>8}{'kürzest':>9}{'Median':>8}{'längst':>8}{'zerschnitten':>14}{'Belege':>9}"
print(kopf)
print("-" * len(kopf))
for name in ["fest 800/100", "Struktur", "LangChain"]:
    w = bewerte(ergebnisse[name])
    print(f"{name:<16}{w['anzahl']:>8}{w['kuerzester']:>9}{w['median']:>8}"
          f"{w['laengster']:>8}{w['zerschnitten']:>14}{str(w['belege']) + '/5':>9}")

print()
print("Ein Chunk aus dem Framework:")
print(ergebnisse["LangChain"][3][:300], "…")

📖 Das Framework kommt auf dieselben Kennzahlen wie die eigene Implementierung. Der Unterschied
liegt in der Anzahl: Der Splitter füllt seine Chunks bis nahe an `chunk_size`, unsere Version
beginnt bei jeder Überschrift neu, sobald die Mindestgröße erreicht ist. Beides ist vertretbar
— weniger, größere Chunks sparen Einträge in der Vector Database, mehr, kleinere Chunks treffen
im Retrieval genauer.

Was das Framework zusätzlich mitbringt: Splitter für HTML, Markdown mit Überschriften als
Metadaten, Code nach Sprache, und `length_function` für Tokens statt Zeichen. Was es nicht
mitbringt, ist die Entscheidung, welche Grenzen für deine Dokumente richtig sind. Die kommt aus
einer Messung wie der oben.

**Für den Rest dieses Kapitels benutzen wir das Struktur-Verfahren.** Seine Chunks beginnen
immer mit einer Überschrift — das ist im Prompt später die halbe Quellenangabe.

---
## 7 · Chunks in die Vector Database

📖 Ein Chunk ist mehr als sein Text. Damit eine Antwort später belegt werden kann, muss an jedem
Chunk stehen, woher er kommt:

| Feld | Beispiel | Wozu |
|---|---|---|
| `chunk_id` | `cve-2026-3224#03` | eindeutige Kennung in der Datenbank |
| `dok_id` | `cve-2026-3224` | aus welchem Dokument |
| `titel` | `CVE-2026-3224 — Authentication Bypass …` | für die Anzeige und den Prompt |
| `quelle` | `wissensbasis/cve-2026-3224.md` | die Datei, für die Quellenangabe |
| `position` | `3` | die wievielte Stelle im Dokument |

Diese Felder heißen in einer Vector Database **Metadaten**. Sie werden nicht eingebettet, aber
mitgespeichert — und man kann nach ihnen filtern, etwa „nur Chunks aus CVE-Advisories".

### 🛠️ Challenge 4: Chunks mit Metadaten

Schreibe `baue_chunk_liste(dokumente)`. Die Funktion chunkt jedes Dokument mit
`chunke_nach_struktur` und gibt eine **flache Liste von Dicts** zurück — ein Dict je Chunk, mit
genau den sechs Feldern aus der Tabelle oben.

Die `chunk_id` ist die Dokumentkennung, ein `#` und die Position mit zwei Stellen:
`f"{dokument['id']}#{position:02d}"`.

*Tipp: `enumerate(...)` liefert Position und Text in einem Durchgang.*

In [ ]:
def baue_chunk_liste(dokumente):
    """Chunkt jedes Dokument und hängt die Metadaten an jeden Chunk."""
    alle = []
    for dokument in dokumente:
        # TODO 1: das Dokument chunken und dabei die Position mitzählen
        for position, text in ...:
            # TODO 2: ein Dict mit den sechs Feldern anhängen
            alle.append(...)
    return alle


In [ ]:
# ✅ Selbsttest
meine_chunks = baue_chunk_liste(dokumente)
FELDER = {"chunk_id", "dok_id", "titel", "quelle", "text", "position"}

assert len(meine_chunks) > 50, f"Zu wenige Chunks: {len(meine_chunks)}"
assert all(set(c) == FELDER for c in meine_chunks), "Jeder Chunk braucht genau die sechs Felder"
assert len({c["chunk_id"] for c in meine_chunks}) == len(meine_chunks), "chunk_id muss eindeutig sein"
assert all(isinstance(c["position"], int) for c in meine_chunks), "position ist eine ganze Zahl"
assert meine_chunks[0]["chunk_id"] == f"{meine_chunks[0]['dok_id']}#00", "Die erste Position ist 00"
assert all(c["text"].strip() for c in meine_chunks), "Kein Chunk darf leer sein"

erste = [c for c in meine_chunks if c["dok_id"] == "cve-2026-3224"]
assert [c["position"] for c in erste] == list(range(len(erste))), "Positionen laufen je Dokument von 0 hoch"

print("✅ Challenge 4 gelöst")
print(f"{len(dokumente)} Dokumente  →  {len(meine_chunks)} Chunks")
print(json.dumps({**erste[1], "text": erste[1]["text"][:80] + " …"}, ensure_ascii=False, indent=2))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_chunk_liste(dokumente):
    """Chunkt jedes Dokument und hängt die Metadaten an jeden Chunk."""
    alle = []
    for dokument in dokumente:
        for position, text in enumerate(chunke_nach_struktur(dokument["text"])):
            alle.append({
                "chunk_id": f"{dokument['id']}#{position:02d}",
                "dok_id": dokument["id"],
                "titel": dokument["titel"],
                "quelle": dokument["quelle"],
                "text": text,
                "position": position,
            })
    return alle
```

Dieselbe Funktion steht in `daten/erzeuge_chunks.py`. Das Skript schreibt damit
`daten/chunks.json` — die Fassung, mit der die folgenden Notebooks starten.

</details>

📖 Jetzt kommen die Chunks in die Vector Database. Wir benutzen **Chroma**, weil es ohne Server
auskommt: Eine Collection ist ein Verzeichnis auf der Platte.

Drei Dinge gehören zu jedem Eintrag:

* `ids` — die `chunk_id`, damit derselbe Chunk nicht zweimal drinsteht,
* `embeddings` — der Vektor zum Text, hier über `helfer.embed()` von `nomic-embed-text`,
* `documents` und `metadatas` — der Text selbst und die Herkunftsfelder.

`hnsw:space` legt fest, wie Chroma zwei Vektoren vergleicht. `cosine` misst den Winkel und
ignoriert die Länge — das übliche Maß für Text-Embeddings.

▶️ Die nächste Zelle legt die Collection an.

In [ ]:
# ▶️ Eine leere, persistente Collection unter daten/chroma/
klient = chromadb.PersistentClient(path=str(helfer.CHROMA_PFAD))

try:
    klient.delete_collection("wissensbasis")
except Exception:
    pass

sammlung = klient.get_or_create_collection("wissensbasis", metadata={"hnsw:space": "cosine"})

print(f"Collection {sammlung.name!r} unter {helfer.CHROMA_PFAD}")
print(f"Einträge: {sammlung.count()}")

### 🛠️ Challenge 5: Chunks in Chroma speichern und suchen

Hier verbindest du vorbereitete Teile. Es geht nicht um die interne Arbeitsweise von Chroma,
sondern um zwei Übergaben:

- schreibe_in_chroma() übergibt IDs, Texte, Embeddings und Metadaten blockweise.
- frage_sammlung() bettet eine Frage ein und übergibt den Vektor an query().

Vervollständige nur die markierten Argumente. Alle Listen eines Schreibblocks müssen dieselbe
Reihenfolge haben. In die Metadaten kommen dok_id, titel, quelle und position.


In [ ]:
def schreibe_in_chroma(sammlung, chunks):
    """Schreibt Chunks mit Embedding und Metadaten in die Collection."""
    for start in range(0, len(chunks), 32):
        block = chunks[start:start + 32]
        texte = [c["text"] for c in block]

        # TODO 1: einen Block in die Collection schreiben
        sammlung.add(
            ids=...,
            embeddings=...,
            documents=...,
            metadatas=...,
        )

    return sammlung.count()


def frage_sammlung(sammlung, frage, n=3):
    """Bettet die Frage ein und holt die n ähnlichsten Chunks."""
    # TODO 2: Frage einbetten und abfragen
    raise NotImplementedError("Challenge 5: frage_sammlung() implementieren")


In [ ]:
# ✅ Selbsttest
anzahl = schreibe_in_chroma(sammlung, meine_chunks)
assert anzahl == len(meine_chunks), f"{len(meine_chunks)} Einträge erwartet, {anzahl} gezählt"

eintrag = sammlung.get(ids=[meine_chunks[7]["chunk_id"]], include=["metadatas", "documents"])
assert set(eintrag["metadatas"][0]) == {"dok_id", "titel", "quelle", "position"}, \
    "Genau diese vier Metadaten gehören an einen Chunk"
assert eintrag["documents"][0] == meine_chunks[7]["text"], "Der Text muss mitgespeichert werden"

treffer = frage_sammlung(sammlung, "Wie schnell muss ein Notfall-Patch eingespielt werden?", n=3)
assert len(treffer["ids"][0]) == 3, "Drei Treffer erwartet"
assert "runbook-patch-management" in [m["dok_id"] for m in treffer["metadatas"][0]], \
    "Das Runbook Patch-Management sollte unter den ersten drei Treffern sein"

print("✅ Challenge 5 gelöst")
print(f"{anzahl} Chunks in der Collection {sammlung.name!r}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def schreibe_in_chroma(sammlung, chunks):
    """Schreibt Chunks mit Embedding und Metadaten in die Collection."""
    for start in range(0, len(chunks), 32):
        block = chunks[start:start + 32]
        texte = [c["text"] for c in block]

        sammlung.add(
            ids=[c["chunk_id"] for c in block],
            embeddings=helfer.embed(texte),
            documents=texte,
            metadatas=[{"dok_id": c["dok_id"], "titel": c["titel"],
                        "quelle": c["quelle"], "position": c["position"]}
                       for c in block],
        )

    return sammlung.count()


def frage_sammlung(sammlung, frage, n=3):
    """Bettet die Frage ein und holt die n ähnlichsten Chunks."""
    return sammlung.query(query_embeddings=helfer.embed([frage]), n_results=n)
```

Die Blöcke von 32 sind kein Detail: Ein einzelner `add`-Aufruf mit tausenden Einträgen läuft in
Zeitüberschreitungen, und jeder Fehler kostet den ganzen Durchgang.

</details>

In [ ]:
# ▶️ Die erste Abfrage
for frage in [f["frage"] for f in fragen[:3]]:
    print(f"❓ {frage}")
    helfer.zeige_treffer(frage_sammlung(sammlung, frage, n=3))
    print()

📖 Das ist Retrieval: Die Frage wird zum Vektor, die Datenbank liefert die Chunks mit dem
kleinsten Abstand. Der Score ist `1 - Distanz`; bei `cosine` liegt er zwischen 0 und 1, und
alles über etwa 0,7 ist bei diesem Embedding-Modell ein brauchbarer Treffer.

Die Collection liegt jetzt in `daten/chroma/` und bleibt dort. Die folgenden Notebooks öffnen
sie mit `helfer.lade_chroma()`, ohne etwas neu zu berechnen.

▶️ Die letzte Zelle gleicht die Collection mit `daten/chunks.json` ab. Damit enthält sie genau
die Chunks, mit denen die anderen Notebooks arbeiten — auch wenn dein Verfahren oben etwas
anders geschnitten hat.

In [ ]:
# ▶️ Die Referenz-Chunks ergänzen, damit die folgenden Notebooks sicher starten
referenz = helfer.lade_chunks()
sammlung = helfer.baue_chroma(referenz, neu=True)

print(f"Collection {sammlung.name!r}: {sammlung.count()} Einträge")
print(f"Pfad: {helfer.CHROMA_PFAD}")

---
## 8 · Was du gebaut hast

* `chunke_fest()` — Chunking nach fester Zeichenlänge mit Überlappung, das Verfahren, das auf
  jedem Text funktioniert und nichts über ihn weiß.
* `chunke_nach_struktur()` — Chunking an Überschriften und Absätzen, mit Mindest- und
  Maximalgröße.
* `bewerte()` — vier Kennzahlen, die den Unterschied messbar machen, statt ihn zu behaupten.
* `baue_chunk_liste()` — Chunks mit `dok_id`, `titel`, `quelle` und `position`.
* `schreibe_in_chroma()` und `frage_sammlung()` — die persistente Collection `wissensbasis` und
  die erste Abfrage darauf.

Die wichtigste Zahl aus dem Vergleich: Das feste Verfahren liefert zwei von fünf Belegstellen
vollständig, das Struktur-Verfahren alle fünf. Chunking entscheidet, was das Retrieval
überhaupt finden kann.

---
### 🔬 Bonus — ohne Lösung

**1. Semantisches Chunking.** Statt an Überschriften wird dort getrennt, wo sich der Inhalt
ändert. Vorgehen:

1. Text in Sätze oder Absätze zerlegen.
2. Jedes Stück mit `helfer.embed()` einbetten.
3. Für jedes benachbarte Paar die Cosine Similarity berechnen.
4. Dort trennen, wo die Ähnlichkeit einen Schwellenwert unterschreitet — üblich ist das
   niedrigste Fünftel aller Abstände.
5. Anschließend dieselbe Mindest- und Maximalgröße durchsetzen wie oben.

Miss das Ergebnis mit `bewerte()` gegen die beiden anderen Verfahren. Interessant ist vor allem
die Systemdokumentation: Findet das Verfahren die Abschnittsgrenzen, die dort ohnehin als
Überschriften stehen?

**2. LLM-basiertes Chunking.** Das Modell entscheidet selbst, wo ein Abschnitt endet. Vorgehen:

1. Dokument in Blöcke von rund 2.000 Zeichen schneiden, an Absatzgrenzen.
2. Je Block einen Prompt mit `helfer.frage_llm()`: die Absätze durchnummeriert vorlegen und
   nach den Nummern fragen, bei denen ein neues Thema beginnt. Antwortformat JSON, damit sich
   das Ergebnis auswerten lässt.
3. An den genannten Stellen trennen, danach die Größengrenzen durchsetzen.

Zwei Fragen dazu: Wie oft liefert `qwen3.5:0.8b` gültiges JSON, und was kostet das Verfahren für die
elf Dokumente an Tokens im Vergleich zu den ersten beiden?

**3. Tokens statt Zeichen.** Setze im `RecursiveCharacterTextSplitter`
`length_function=helfer.zaehle_tokens` und `chunk_size=300`. Welche Dokumente ändern sich am
stärksten — und warum gerade die mit den Tabellen?